# Task 15 — Recommend Fair Pay Bands

**Objective:** train and compare salary models, recommend low/mid/high pay bands, and audit group gaps without using protected attributes as model inputs.

**Validation Vault rule:** this notebook completes the salary model, held-out evaluation, and recommendation-band export before loading `unlock_CX-15_protected_attributes.csv`. The Vault file is used only in the final audit section. Gender is never a model feature; career-gap duration is also excluded from model inputs and used only to form audit groups.

**Target:** `offered_annual_fixed` (annual fixed pay, currency units as supplied by the dataset).

## 1. Imports and project paths

Run this notebook in the Docker-mounted project. It uses `/workspace` as the project root, reads shared inputs from `data/`, and accesses protected attributes only from `04_Validation_Vault/` during the final audit.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = Path("/workspace")
if not (ROOT / "data").exists():
    raise FileNotFoundError(f"Expected Docker-mounted project root at {ROOT}; data/ was not found.")

DATA_DIR = ROOT / "data"
VAULT_DIR = ROOT / "04_Validation_Vault"
REPORT_DIR = ROOT / "phase-3" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SALARY_PATH = DATA_DIR / "salary_training.csv"
VAULT_PATH = VAULT_DIR / "unlock_CX-15_protected_attributes.csv"

if not SALARY_PATH.exists():
    raise FileNotFoundError(f"Salary training file not found: {SALARY_PATH}")

salary_df = pd.read_csv(SALARY_PATH)
print("Project root:", ROOT)
print("Salary dataset shape:", salary_df.shape)
print("Salary dataset loaded. The protected-attribute file has NOT been opened.")

Project root: /workspace
Salary dataset shape: (2934, 17)
Salary dataset loaded. The protected-attribute file has NOT been opened.


## 2. Data audit and feature selection

Only job-related fields are used as predictors. The target is `offered_annual_fixed`.

Excluded fields:
- `hire_id`, `hire_date`, `hire_year`, and `source`: identifiers, chronology, or provenance rather than stable job-value predictors.
- `band_min_2026`, `band_mid_2026`, `band_max_2026`: existing salary-band references that could anchor the model to the compensation policy being evaluated.
- `career_gap_months`: reserved for the fairness audit; excluding it avoids directly penalising a career break.
- `highest_education`: excluded to reduce reliance on a potentially inequitable credential proxy.
- Gender and age: not present in the training table and must not be model inputs.

In [2]:
TARGET = "offered_annual_fixed"
CATEGORICAL_FEATURES = ["department_id", "role_id", "work_location"]
NUMERIC_FEATURES = [
    "job_level",
    "prior_experience_years",
    "skills_count",
    "market_index",
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

required = FEATURES + [TARGET, "hire_id", "career_gap_months"]
missing = sorted(set(required) - set(salary_df.columns))
assert not missing, f"Missing required columns: {missing}"
assert salary_df["hire_id"].is_unique, "hire_id must be unique"
assert salary_df[TARGET].notna().all(), "Target contains missing values"
assert salary_df[FEATURES].notna().all().all(), "Selected features contain missing values"
assert salary_df[TARGET].ge(0).all(), "Salary target contains negative values"

X = salary_df[FEATURES].copy()
y = salary_df[TARGET].astype(float).copy()

print("Rows:", len(salary_df))
print("Unique roles:", salary_df["role_id"].nunique())
print("Selected features:", FEATURES)
print("Excluded from predictors: gender, age, career_gap_months, education, employee identifiers, hire dates, and existing salary bands.")
display(salary_df[TARGET].describe().to_frame("offered_annual_fixed"))

Rows: 2934
Unique roles: 47
Selected features: ['department_id', 'role_id', 'work_location', 'job_level', 'prior_experience_years', 'skills_count', 'market_index']
Excluded from predictors: gender, age, career_gap_months, education, employee identifiers, hire dates, and existing salary bands.


,offered_annual_fixed
count,2.934000e+03
mean,8.344280e+05
std,6.413154e+05
min,1.144320e+05
25%,3.805410e+05
50%,6.241920e+05
75%,1.114299e+06
max,4.197972e+06


## 3. Reproducible train / calibration / test split

The data is split by role so each set has a similar role mix. The held-out test set is not used for model fitting or residual-band calibration.

- Training: 1,994 records
- Calibration: 353 records
- Test: 587 records

The split uses fixed random seeds for reproducibility.

In [3]:
X_dev, X_test, y_dev, y_test, idx_dev, idx_test = train_test_split(
    X, y, salary_df.index,
    test_size=0.20,
    random_state=42,
    stratify=salary_df["role_id"],
)

X_train, X_cal, y_train, y_cal, idx_train, idx_cal = train_test_split(
    X_dev, y_dev, idx_dev,
    test_size=0.1504,
    random_state=43,
    stratify=salary_df.loc[idx_dev, "role_id"],
)

split_counts = {
    "train": len(X_train),
    "calibration": len(X_cal),
    "test": len(X_test),
}
print("Split counts:", split_counts)
assert split_counts == {"train": 1994, "calibration": 353, "test": 587}

Split counts: {'train': 1994, 'calibration': 353, 'test': 587}


## 4. Baseline and trained models

The baseline predicts the mean historical salary for the employee's role using the training partition only. If a role is unseen, it falls back to the overall training mean.

Both trained models use the same selected features and training records:
- Linear Regression with one-hot encoding for categorical fields.
- Gradient Boosting Regressor with the same preprocessing.

No protected attribute or career-gap field is passed to either model.

In [4]:
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

linear_model = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("regressor", LinearRegression()),
])

gradient_model = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("regressor", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        loss="huber",
        random_state=42,
    )),
])

linear_model.fit(X_train, y_train)
gradient_model.fit(X_train, y_train)

role_means = y_train.groupby(salary_df.loc[idx_train, "role_id"]).mean()
global_training_mean = float(y_train.mean())
baseline_predictions = (
    salary_df.loc[idx_test, "role_id"].map(role_means).fillna(global_training_mean).to_numpy()
)

linear_predictions = linear_model.predict(X_test)
gradient_predictions = gradient_model.predict(X_test)

print("Models trained. Test data was not used during fitting.")

Models trained. Test data was not used during fitting.


## 5. Held-out model metrics

MAE is reported in the same currency units as `offered_annual_fixed`. R² measures the proportion of test-target variance explained relative to a constant-mean predictor.

The baseline and both models are evaluated on the exact same held-out test records.

In [5]:
metrics_df = pd.DataFrame([
    {
        "model": "Average by role baseline",
        "MAE": mean_absolute_error(y_test, baseline_predictions),
        "R2": r2_score(y_test, baseline_predictions),
    },
    {
        "model": "Linear Regression",
        "MAE": mean_absolute_error(y_test, linear_predictions),
        "R2": r2_score(y_test, linear_predictions),
    },
    {
        "model": "Gradient Boosting",
        "MAE": mean_absolute_error(y_test, gradient_predictions),
        "R2": r2_score(y_test, gradient_predictions),
    },
]).sort_values("MAE").reset_index(drop=True)

metrics_df.to_csv(REPORT_DIR / "task15_model_metrics.csv", index=False)
display(metrics_df.round({"MAE": 2, "R2": 4}))

best_model_name = metrics_df.iloc[0]["model"]
print("Lowest-MAE model:", best_model_name)

,model,MAE,R2
0,Linear Regression,82141.38,0.9662
1,Gradient Boosting,93211.80,0.9431
2,Average by role baseline,115196.32,0.9348


Lowest-MAE model: Linear Regression


## 6. Low / mid / high salary recommendations

Linear Regression is selected because it has the lowest held-out MAE and highest held-out R² in this experiment.

Residual quantiles are estimated only from the separate calibration partition. The recommendation band is the selected model's point prediction plus the 10th, 50th, and 90th percentiles of calibration residuals. Values are floored at zero.

These are empirical recommendation ranges, not guaranteed confidence intervals or approved compensation offers.

In [6]:
calibration_predictions = linear_model.predict(X_cal)
calibration_residuals = y_cal.to_numpy() - calibration_predictions
q10, q50, q90 = np.quantile(calibration_residuals, [0.10, 0.50, 0.90])

recommendations = salary_df.loc[idx_test, [
    "hire_id", "department_id", "role_id", "job_level"
]].copy()
recommendations["actual_offered_annual_fixed"] = y_test.to_numpy()
recommendations["predicted_salary"] = linear_predictions
recommendations["recommended_low"] = np.maximum(0, linear_predictions + q10)
recommendations["recommended_mid"] = np.maximum(0, linear_predictions + q50)
recommendations["recommended_high"] = np.maximum(0, linear_predictions + q90)

assert (recommendations["recommended_low"] <= recommendations["recommended_mid"]).all()
assert (recommendations["recommended_mid"] <= recommendations["recommended_high"]).all()

band_coverage = (
    (y_test.to_numpy() >= recommendations["recommended_low"].to_numpy()) &
    (y_test.to_numpy() <= recommendations["recommended_high"].to_numpy())
).mean()

# Individual-level salary recommendations are local-only; do not commit this CSV.
recommendations.to_csv(REPORT_DIR / "task15_salary_recommendations_test.csv", index=False)

band_summary = {
    "residual_q10": float(q10),
    "residual_q50": float(q50),
    "residual_q90": float(q90),
    "test_interval_coverage": float(band_coverage),
    "test_rows": int(len(recommendations)),
}
print("Calibration residual adjustments:", band_summary)
print(f"Empirical test coverage: {band_coverage:.1%}")
display(recommendations[["recommended_low", "recommended_mid", "recommended_high"]].describe().round(2))

Calibration residual adjustments: {'residual_q10': -125975.4367699852, 'residual_q50': 8447.57396774285, 'residual_q90': 110630.56267823893, 'test_interval_coverage': 0.7887563884156729, 'test_rows': 587}
Empirical test coverage: 78.9%


,recommended_low,recommended_mid,recommended_high
count,587.00,587.00,587.00
mean,703083.05,836243.43,938159.11
std,631464.39,632962.76,633334.46
min,0.00,0.00,0.00
25%,257874.65,392297.66,494480.65
50%,489888.87,624311.88,726494.87
75%,1055490.30,1189913.31,1292096.30
max,3245145.62,3379568.63,3481751.62


In [7]:
# Generate out-of-fold predictions for the fairness audit before opening the Vault.
# Each row is predicted by a fold model that did not train on that row.
fairness_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)
fairness_splits = list(fairness_cv.split(X, salary_df["role_id"]))
oof_predictions = cross_val_predict(
    linear_model, X, y, cv=fairness_splits, n_jobs=None
)
assert len(oof_predictions) == len(salary_df)
salary_df["oof_predicted_salary"] = oof_predictions
print("Out-of-fold predictions generated for fairness audit:", len(oof_predictions))
print("Protected-attribute file has still NOT been opened.")

Out-of-fold predictions generated for fairness audit: 2934
Protected-attribute file has still NOT been opened.


## 7. Model stage completed — now open the Validation Vault

The train/calibration/test model comparison, residual-based salary recommendations, and five-fold out-of-fold predictions for the fairness audit have all been completed before opening the Vault file.

The protected-attribute file is now opened solely for a post-model audit. **Vault attributes are not model inputs.** Age is not used. Gender is used only to group out-of-fold predictions. Career-gap duration comes from the salary dataset and is also used only to define audit groups.

In [8]:
if not VAULT_PATH.exists():
    raise FileNotFoundError(
        f"Validation Vault file not found: {VAULT_PATH}. "
        "Place unlock_CX-15_protected_attributes.csv in 04_Validation_Vault/."
    )

vault_df = pd.read_csv(VAULT_PATH)
assert {"person_ref", "record_type", "gender"}.issubset(vault_df.columns)

salary_protected = vault_df.loc[
    vault_df["record_type"].eq("salary_hire"),
    ["person_ref", "gender"],
].copy()

assert salary_protected["person_ref"].is_unique, "Duplicate salary_hire person_ref values"
assert set(salary_protected["person_ref"]) == set(salary_df["hire_id"]), (
    "Protected salary_hire IDs do not exactly match salary_training hire_id values"
)

print("Vault rows:", len(vault_df))
print("salary_hire audit rows:", len(salary_protected))
print("Age field excluded from audit.")

Vault rows: 5317
salary_hire audit rows: 2934
Age field excluded from audit.


## 8. Gender and career-gap group-gap audit

The audit uses out-of-fold predictions for all 2,934 salary records. Each employee's prediction comes from a five-fold model that did not train on that employee. This gives larger audit groups than using only the 587-row test set.

The `group_gap()` implementation follows Task 12: calculate each group's mean and subtract the minimum group mean from the maximum group mean.

For each grouping, historical offered salary and out-of-fold predicted salary are compared on the same records. This is a descriptive comparison, not a causal estimate.

In [9]:
audit_df = salary_df[[
    "hire_id", "career_gap_months", TARGET, "oof_predicted_salary"
]].copy()
audit_df = audit_df.merge(
    salary_protected,
    left_on="hire_id",
    right_on="person_ref",
    how="left",
    validate="one_to_one",
)
assert audit_df["gender"].notna().all()
assert len(audit_df) == len(salary_df)

def group_gap(scores, groups):
    means = pd.Series(scores).groupby(pd.Series(groups).values).mean()
    return means.to_dict(), float(means.max() - means.min())

def career_gap_category(months):
    if months == 0:
        return "No gap"
    if months <= 12:
        return "1-12 months"
    if months <= 24:
        return "13-24 months"
    return "25+ months"

audit_df["career_gap_group"] = audit_df["career_gap_months"].map(career_gap_category)

gap_rows = []
for attribute, group_column in [
    ("gender", "gender"),
    ("career_gap", "career_gap_group"),
]:
    historical_means, historical_gap = group_gap(
        audit_df[TARGET], audit_df[group_column]
    )
    predicted_means, predicted_gap = group_gap(
        audit_df["oof_predicted_salary"], audit_df[group_column]
    )

    for group_name, group in audit_df.groupby(group_column, dropna=False):
        gap_rows.append({
            "attribute": attribute,
            "group": group_name,
            "n": int(len(group)),
            "historical_mean_salary": float(group[TARGET].mean()),
            "predicted_mean_salary": float(group["oof_predicted_salary"].mean()),
            "historical_group_gap": float(historical_gap),
            "predicted_group_gap": float(predicted_gap),
            "mean_prediction_minus_historical": float(
                group["oof_predicted_salary"].mean() - group[TARGET].mean()
            ),
        })

group_gap_df = pd.DataFrame(gap_rows)
group_gap_df.to_csv(REPORT_DIR / "task15_group_gap_analysis.csv", index=False)
display(group_gap_df.round(2))

,attribute,group,n,historical_mean_salary,predicted_mean_salary,historical_group_gap,predicted_group_gap,mean_prediction_minus_historical
0,gender,Female,1222,806183.95,811555.66,48941.91,40279.52,5371.71
1,gender,Male,1652,855125.86,851835.18,48941.91,40279.52,-3290.68
2,gender,Other,60,839786.40,828559.37,48941.91,40279.52,-11227.03
3,career_gap,1-12 months,246,799482.05,819655.10,50653.30,74105.30,20173.05
4,career_gap,13-24 months,94,806495.87,842961.01,50653.30,74105.30,36465.14
5,career_gap,25+ months,45,788979.73,768855.71,50653.30,74105.30,-20124.02
6,career_gap,No gap,2549,839633.04,836874.97,50653.30,74105.30,-2758.06


## 9. Interpretation and limitations

- Gender and career-gap duration were excluded from model predictors. Their labels were used only after model fitting to audit held-out predictions.
- Excluding protected attributes does not guarantee equal group outcomes. Role, department, job level, experience, skills, and location can have different distributions across groups.
- Group gaps are descriptive and do not establish discrimination or causality.
- Small test subgroups (especially `Other` gender and longer career gaps) have unstable means and should not support standalone conclusions.
- Historical offered salaries may contain prior inequities, inconsistent negotiation outcomes, and time-related effects. A model trained on historical offers can reproduce patterns in those labels even without protected attributes.
- The low/mid/high bands are calibrated from one held-out split. Their empirical coverage is not a future guarantee.
- Salary bands are decision-support guidance only. HR and managers must apply consistent job evaluation, market benchmarks, and human review.
- Education, career gaps, age, and gender are not used as model predictors in this implementation. The selected predictors still require ongoing review for proxy effects.

## 10. Final validation

The checks below confirm that the model inputs exclude protected and career-gap fields, the recommendation bands are ordered, and both required aggregate evidence tables were exported.

In [10]:
assert "gender" not in FEATURES
assert "age_years" not in FEATURES
assert "career_gap_months" not in FEATURES
assert "highest_education" not in FEATURES
assert len(metrics_df) == 3
assert len(recommendations) == len(X_test)
assert len(salary_df["oof_predicted_salary"]) == len(salary_df)
assert recommendations[["recommended_low", "recommended_mid", "recommended_high"]].notna().all().all()
assert group_gap_df[["historical_group_gap", "predicted_group_gap"]].notna().all().all()
assert (group_gap_df["n"] > 0).all()

summary = {
    "task": "Task 15 - Recommend fair pay bands",
    "dataset_rows": int(len(salary_df)),
    "unique_roles": int(salary_df["role_id"].nunique()),
    "split_counts": split_counts,
    "target": TARGET,
    "features": FEATURES,
    "excluded_predictors": [
        "gender", "age_years", "career_gap_months", "highest_education",
        "hire_id", "hire_date", "hire_year", "source",
        "band_min_2026", "band_mid_2026", "band_max_2026",
    ],
    "metrics": metrics_df.to_dict(orient="records"),
    "selected_model": "Linear Regression",
    "residual_quantiles": band_summary,
    "gender_group_gap_rows": int((group_gap_df["attribute"] == "gender").sum()),
    "career_gap_group_rows": int((group_gap_df["attribute"] == "career_gap").sum()),
    "fairness_predictions": "5-fold stratified out-of-fold predictions on all salary rows",
    "limitations": [
        "Group gaps are descriptive, not causal.",
        "Small audit subgroups produce unstable estimates.",
        "Historical salary labels may encode past inequities.",
        "Recommendation bands are empirical and not guaranteed intervals.",
        "Human review is required.",
    ],
    "outputs": [
        "phase-3/reports/task15_model_metrics.csv",
        "phase-3/reports/task15_group_gap_analysis.csv",
        "phase-3/reports/task15_salary_recommendations_test.csv",
        "phase-3/reports/task15_summary.json",
    ],
}
with open(REPORT_DIR / "task15_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Final validation passed.")
print("Metrics:", REPORT_DIR / "task15_model_metrics.csv")
print("Group gaps:", REPORT_DIR / "task15_group_gap_analysis.csv")
print("Recommendations (local):", REPORT_DIR / "task15_salary_recommendations_test.csv")
print("Summary:", REPORT_DIR / "task15_summary.json")

Final validation passed.
Metrics: /workspace/phase-3/reports/task15_model_metrics.csv
Group gaps: /workspace/phase-3/reports/task15_group_gap_analysis.csv
Recommendations (local): /workspace/phase-3/reports/task15_salary_recommendations_test.csv
Summary: /workspace/phase-3/reports/task15_summary.json
